# 01 Major Index Sliding Window Pipeline

This notebook converts the feature panel into the shared rolling-window bundle used by the benchmark, machine-learning, deep-learning, and quantum notebooks.

## Locked Design
- Forecast target: one-step-ahead daily return
- Window sizes: `10` and `20`
- Test point: the final observation in each window
- Effective train/test structure: `9/1` for `w=10`, `19/1` for `w=20`
- Hyperparameter tuning: windows whose test-point date falls in `2022`
- Evaluation: windows whose test-point date falls in `2023-01-01` to `2026-03-31`
- No additional train/validation/test split beyond this chronological rule

## Confirmed Inputs
- Indices: `NASDAQ`, `SP500`, `DOWJONES`, `RUSSELL3000`
- Features: the same nine inputs used across every downstream model


In [ ]:
from __future__ import annotations

import time
from contextlib import contextmanager
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display


def resolve_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "requirements.txt").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Run this notebook from the project root or its notebooks directory.")


PROJECT_ROOT = resolve_project_root()
INPUT_BUNDLE_PATH = PROJECT_ROOT / "outputs" / "00_major_index_data_collection_and_eda" / "major_index_data_eda_bundle.joblib"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "01_major_index_sliding_window_pipeline"
TABLE_DIR = OUTPUT_ROOT / "tables"
for directory in [OUTPUT_ROOT, TABLE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

START_DATE = "2022-01-01"
END_DATE = "2026-03-31"
TUNING_END_DATE = "2022-12-31"
LOOKBACK_WINDOWS = [10, 20]
TEST_RATIO_BY_WINDOW = {10: 0.10, 20: 0.05}
STEP = 1
TARGET_COL = "target_close"
FEATURE_COLS = ["rsi14", "macd_hist", "bb_z20", "roc10", "vol20", "sma10_gap", "atr14_pct", "williams_r14"]
REQUESTED_INDICES = ["NASDAQ", "SP500", "DOWJONES", "RUSSELL3000"]

BUNDLE_PATH = OUTPUT_ROOT / "major_index_sliding_window_bundle.joblib"
CASE_SUMMARY_PATH = TABLE_DIR / "case_summary.csv"
WINDOW_MANIFEST_PATH = TABLE_DIR / "window_manifest.csv"
TIMING_SUMMARY_PATH = TABLE_DIR / "timing_summary.csv"

TIMING_ROWS: list[dict] = []


@contextmanager
def timed_step(step_name: str):
    start = time.perf_counter()
    yield
    elapsed_seconds = time.perf_counter() - start
    TIMING_ROWS.append({"step": step_name, "elapsed_seconds": elapsed_seconds})
    print(f"[TIMER] {step_name}: {elapsed_seconds:.2f}s")


In [ ]:
def load_feature_panel(path: Path = INPUT_BUNDLE_PATH) -> pd.DataFrame:
    bundle = joblib.load(path)
    frame = bundle["feature_panel"].copy()
    frame["Date"] = pd.to_datetime(frame["Date"], errors="coerce")
    return frame.sort_values(["index_key", "Date"]).reset_index(drop=True)


def build_case_frame(feature_panel: pd.DataFrame, index_key: str) -> pd.DataFrame:
    frame = (
        feature_panel.loc[feature_panel["index_key"].eq(index_key), ["Date", "index_key", "index_label", "close", *FEATURE_COLS]]
        .copy()
        .sort_values("Date")
        .reset_index(drop=True)
    )
    frame["reference_close"] = pd.to_numeric(frame["close"], errors="coerce")
    frame = frame.drop(columns=["close"])
    frame["target_date"] = frame["Date"].shift(-1)
    frame["target_close"] = frame["reference_close"].shift(-1)
    frame[TARGET_COL] = frame["target_close"]
    frame = frame.loc[(frame["Date"] >= pd.Timestamp(START_DATE)) & (frame["Date"] <= pd.Timestamp(END_DATE))].copy()
    frame = frame.dropna(subset=FEATURE_COLS + [TARGET_COL, "target_date", "target_close"]).reset_index(drop=True)
    return frame


def iter_windows_plus1(df: pd.DataFrame, window_size: int, test_ratio: float, step: int = 1):
    test_len = max(1, int(round(window_size * test_ratio)))
    n = len(df)
    for start in range(0, n - window_size + 1, step):
        win = df.iloc[start : start + window_size]
        yield {
            "start": start,
            "end_exclusive": start + window_size,
            "test_len": test_len,
            "train_df": win.iloc[: window_size - test_len],
            "test_df": win.iloc[window_size - test_len :],
        }


def build_window_manifest(df: pd.DataFrame, window_size: int, test_ratio: float, step: int = 1) -> pd.DataFrame:
    rows = []
    for window_id, win in enumerate(iter_windows_plus1(df, window_size, test_ratio, step=step), start=1):
        train_df = win["train_df"]
        test_df = win["test_df"]
        test_date = pd.Timestamp(test_df["target_date"].iloc[-1])
        rows.append(
            {
                "window_id": int(window_id),
                "window_size": int(window_size),
                "test_ratio": float(test_ratio),
                "start": int(win["start"]),
                "end_exclusive": int(win["end_exclusive"]),
                "train_rows": int(len(train_df)),
                "test_rows": int(len(test_df)),
                "train_start_date": pd.Timestamp(train_df["Date"].iloc[0]),
                "train_end_date": pd.Timestamp(train_df["Date"].iloc[-1]),
                "test_reference_date": pd.Timestamp(test_df["Date"].iloc[-1]),
                "test_target_date": test_date,
                "is_tuning_window": bool(test_date <= pd.Timestamp(TUNING_END_DATE)),
            }
        )
    return pd.DataFrame(rows)


In [ ]:
with timed_step("load_and_prepare_case_frames"):
    feature_panel = load_feature_panel()
    case_frames: dict[str, pd.DataFrame] = {}
    index_rows = []
    for index_key in REQUESTED_INDICES:
        case_frame = build_case_frame(feature_panel, index_key)
        case_frames[index_key] = case_frame
        index_rows.append(
            {
                "index_key": index_key,
                "rows_after_feature_cleanup": int(len(case_frame)),
                "first_reference_date": pd.Timestamp(case_frame["Date"].min()),
                "last_reference_date": pd.Timestamp(case_frame["Date"].max()),
                "first_target_date": pd.Timestamp(case_frame["target_date"].min()),
                "last_target_date": pd.Timestamp(case_frame["target_date"].max()),
            }
        )
    index_frame_summary = pd.DataFrame(index_rows).sort_values("index_key").reset_index(drop=True)

print("Per-index prepared frame summary")
display(index_frame_summary)


In [ ]:
with timed_step("build_window_manifests_and_bundle"):
    case_bundle: dict[str, dict] = {}
    summary_rows = []
    manifest_frames = []

    for index_key, frame in case_frames.items():
        for window_size in LOOKBACK_WINDOWS:
            case_key = f"{index_key}_w{int(window_size)}"
            test_ratio = float(TEST_RATIO_BY_WINDOW[int(window_size)])
            window_manifest = build_window_manifest(frame, window_size=window_size, test_ratio=test_ratio, step=STEP)
            tuning_windows = int(window_manifest["is_tuning_window"].sum())
            eval_windows = int((~window_manifest["is_tuning_window"]).sum())

            case_bundle[case_key] = {
                "case_key": case_key,
                "index_key": index_key,
                "window_size": int(window_size),
                "test_ratio": float(test_ratio),
                "feature_cols": FEATURE_COLS,
                "target_col": TARGET_COL,
                "df": frame.copy(),
                "window_manifest": window_manifest.copy(),
            }

            manifest_export = window_manifest.copy()
            manifest_export["case_key"] = case_key
            manifest_export["index_key"] = index_key
            manifest_frames.append(manifest_export)

            summary_rows.append(
                {
                    "case_key": case_key,
                    "index_key": index_key,
                    "window_size": int(window_size),
                    "samples": int(len(frame)),
                    "flat_feature_count": int(len(FEATURE_COLS)),
                    "n_windows": int(len(window_manifest)),
                    "tuning_windows": tuning_windows,
                    "eval_windows": eval_windows,
                    "first_test_target_date": pd.Timestamp(window_manifest["test_target_date"].min()),
                    "last_test_target_date": pd.Timestamp(window_manifest["test_target_date"].max()),
                }
            )

    case_summary = pd.DataFrame(summary_rows).sort_values(["index_key", "window_size"]).reset_index(drop=True)
    window_manifest_all = pd.concat(manifest_frames, ignore_index=True).sort_values(["case_key", "window_id"]).reset_index(drop=True)

    bundle = {
        "config": {
            "start_date": START_DATE,
            "end_date": END_DATE,
            "tuning_end_date": TUNING_END_DATE,
            "lookback_windows": LOOKBACK_WINDOWS,
            "test_ratio_by_window": TEST_RATIO_BY_WINDOW,
            "step": STEP,
            "feature_cols": FEATURE_COLS,
            "target_col": TARGET_COL,
            "target_definition": "One-step-ahead index level.",
            "workflow_note": "Plus-one sliding windows with the final observation held out as the test point.",
        },
        "case_bundle": case_bundle,
        "case_summary": case_summary,
        "window_manifest_all": window_manifest_all,
    }
    joblib.dump(bundle, BUNDLE_PATH)
    case_summary.to_csv(CASE_SUMMARY_PATH, index=False)
    window_manifest_all.to_csv(WINDOW_MANIFEST_PATH, index=False)

timing_summary = pd.DataFrame(TIMING_ROWS).sort_values("elapsed_seconds", ascending=False).reset_index(drop=True)
timing_summary.to_csv(TIMING_SUMMARY_PATH, index=False)

print("Case summary")
display(case_summary)
print("Window manifest preview")
display(window_manifest_all.head(20))
print("Timing summary")
display(timing_summary)
print("Rolling-window bundle prepared")
